# Correlated-feature cleaning

Creates `train.pkl`, `validation.pkl`, and `test.pkl` from the labeled fit, tune, and validation splits. Correlations are selected on fit rows before preprocessing.

In [4]:
import pickle
import numpy as np
from tqdm.auto import tqdm

from p1.cache import cached_raw, cached_clean
from p1.cleaning import SCHEMA, clean_column
from p1.config import INTERIM_DATA_DIR, RAW_DATA_DIR
from p1.training import split_rows

CACHE = INTERIM_DATA_DIR / 'fd_cleaning_correlated_sub_sample'
CACHE.mkdir(parents=True, exist_ok=True)
SUB_SAMPLE = True
x_train, x_public_test, y_raw, train_ids, test_ids, names = cached_raw(
    RAW_DATA_DIR, CACHE, sub_sample=SUB_SAMPLE
)
y = (y_raw == 1).astype(int)
fit_idx, tune_idx, valid_idx = split_rows(y, seed=2026)
x_fit = x_train[fit_idx]

MANUAL_DROP = {
    'SEQNO': 'record sequence; high-cardinality code',
    '_PSU': 'identical to SEQNO',
    'IDATE': 'reconstructible from IMONTH/IDAY/IYEAR',
}
THRESHOLD = 0.7
print(f'Fit rows: {len(fit_idx)}; raw fields: {len(names)}')

Raw data: cached
Fit rows: 5249; raw fields: 321


In [5]:
def normalized_mi(left, right):
    pairs, counts = np.unique(np.column_stack((left, right)), axis=0, return_counts=True)
    _, counts_x = np.unique(left, return_counts=True)
    _, counts_y = np.unique(right, return_counts=True)
    n = len(left)
    mi = 0.0
    x_counts = dict(zip(np.unique(left).tolist(), counts_x.tolist()))
    y_counts = dict(zip(np.unique(right).tolist(), counts_y.tolist()))
    for (x_value, y_value), count in zip(pairs, counts):
        pxy = count / n
        mi += pxy * np.log(pxy / ((x_counts[x_value] / n) * (y_counts[y_value] / n)))
    hx = -sum((count / n) * np.log(count / n) for count in counts_x)
    hy = -sum((count / n) * np.log(count / n) for count in counts_y)
    return 0.0 if hx == 0 or hy == 0 else float(mi / np.sqrt(hx * hy))

def categorical_values(values, name):
    values, unavailable, _ = clean_column(values, SCHEMA[name])
    return np.where(unavailable, -1, values).astype(np.int64)

prepared = {}
for j, name in enumerate(names):
    if name in MANUAL_DROP:
        continue
    values, unavailable, _ = clean_column(x_fit[:, j], SCHEMA[name])
    if unavailable.mean() > 0.5 or unavailable.mean() == 1:
        continue
    prepared[name] = (values, unavailable, SCHEMA[name]['kind'])

to_drop = dict(MANUAL_DROP)
numeric_names = [n for n, (_, _, k) in prepared.items() if k == 'numeric']
categorical_names = [n for n, (_, _, k) in prepared.items() if k == 'categorical']

numeric_pair_count = len(numeric_names) * (len(numeric_names) - 1) // 2
with tqdm(total=numeric_pair_count, desc='Pearson pairs') as progress:
    for i, left_name in enumerate(numeric_names):
        left = prepared[left_name][0]
        for right_name in numeric_names[i + 1:]:
            right = prepared[right_name][0]
            mask = np.isfinite(left) & np.isfinite(right)
            if mask.sum() and abs(np.corrcoef(left[mask], right[mask])[0, 1]) > THRESHOLD:
                to_drop[right_name] = f'|Pearson r| > {THRESHOLD} with {left_name}'
            progress.update(1)

encoded = {n: categorical_values(x_fit[:, names.index(n)], n) for n in categorical_names}
categorical_pair_count = len(categorical_names) * (len(categorical_names) - 1) // 2
with tqdm(total=categorical_pair_count, desc='NMI pairs') as progress:
    for i, left_name in enumerate(categorical_names):
        for right_name in categorical_names[i + 1:]:
            score = normalized_mi(encoded[left_name], encoded[right_name])
            if score > THRESHOLD:
                to_drop[right_name] = f'NMI > {THRESHOLD} with {left_name}'
            progress.update(1)

print(f'Numeric fields: {len(numeric_names)}')
print(f'Categorical fields: {len(categorical_names)}')
print(f'Fields selected for removal: {len(to_drop)}')
print(to_drop)

Pearson pairs:   0%|          | 0/990 [00:00<?, ?it/s]

/home/mpj/repositories/epfl_ml_p1/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3037: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/home/mpj/repositories/epfl_ml_p1/.venv/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3038: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
NMI pairs: 100%|██████████| 7381/7381 [00:15<00:00, 489.57it/s]

Numeric fields: 45
Categorical fields: 122
Fields selected for removal: 60
{'SEQNO': 'record sequence; high-cardinality code', '_PSU': 'identical to SEQNO', 'IDATE': 'reconstructible from IMONTH/IDAY/IYEAR', 'NUMMEN': '|Pearson r| > 0.7 with NUMADULT', '_RAWRAKE': '|Pearson r| > 0.7 with NUMMEN', '_WT2RAKE': '|Pearson r| > 0.7 with _STRWT', '_LLCPWT': '|Pearson r| > 0.7 with _WT2RAKE', 'MAXVO2_': '|Pearson r| > 0.7 with _AGE80', 'FC60_': '|Pearson r| > 0.7 with MAXVO2_', 'HTM4': '|Pearson r| > 0.7 with HTIN4', '_BMI5': '|Pearson r| > 0.7 with WTKG3', 'ORNGDAY_': '|Pearson r| > 0.7 with FTJUDA1_', 'VEGEDA1_': '|Pearson r| > 0.7 with ORNGDAY_', '_FRUTSUM': '|Pearson r| > 0.7 with VEGEDA1_', '_VEGESUM': '|Pearson r| > 0.7 with _FRUTSUM', '_MISVEGN': '|Pearson r| > 0.7 with _MISFRTN', 'PAMIN11_': '|Pearson r| > 0.7 with _MINAC11', 'PA1MIN_': '|Pearson r| > 0.7 with PAMIN21_', 'PAMIN21_': '|Pearson r| > 0.7 with _MINAC21', 'PAVIG11_': '|Pearson r| > 0.7 with PAMIN11_', 'PAVIG21_': '|Pearson

In [6]:
prep, X_fit, X_tune, X_valid, _ = cached_clean(
    x_train, x_public_test, names, (fit_idx, tune_idx, valid_idx),
    RAW_DATA_DIR, CACHE, max_missing=0.50, manual_drop=to_drop,
)

outputs = {'train.pkl': X_fit, 'validation.pkl': X_tune, 'test.pkl': X_valid}
for filename, matrix in outputs.items():
    with (CACHE / filename).open('wb') as handle:
        pickle.dump(matrix, handle, protocol=pickle.HIGHEST_PROTOCOL)

print(f'Saved {X_fit.shape[1]} processed features')
for filename in outputs:
    print(CACHE / filename)

Cleaned matrices: cached
Saved 1349 processed features
/home/mpj/repositories/epfl_ml_p1/data/interim/fd_cleaning_correlated_sub_sample/train.pkl
/home/mpj/repositories/epfl_ml_p1/data/interim/fd_cleaning_correlated_sub_sample/validation.pkl
/home/mpj/repositories/epfl_ml_p1/data/interim/fd_cleaning_correlated_sub_sample/test.pkl
